# Gold: Citi Bike Station Activity (v1) — SQL

SQL-cell equivalent of `gold_citibike_station_activity_v1.ipynb`, using
`%%sql` magic instead of the DataFrame API. Same result: reads the
`citibike.tripdata` bronze table and persists
`citibike.gold_station_activity_v1`.

Requires the bronze table to already exist — run
`data-generators/bronze_ingest_citibike.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: station activity summary

Aggregates by start station: total rides, member vs. casual ride counts,
and average trip duration in minutes. `CREATE OR REPLACE TABLE` keeps this
re-runnable.

In [ ]:
%%sql
CREATE OR REPLACE TABLE citibike.gold_station_activity_v1 AS
SELECT
    start_station_name,
    COUNT(*) AS total_rides,
    SUM(CASE WHEN member_casual = 'member' THEN 1 ELSE 0 END) AS member_rides,
    SUM(CASE WHEN member_casual = 'casual' THEN 1 ELSE 0 END) AS casual_rides,
    ROUND(AVG((unix_timestamp(ended_at) - unix_timestamp(started_at)) / 60.0), 2) AS avg_duration_minutes
FROM citibike.tripdata
WHERE start_station_name IS NOT NULL
GROUP BY start_station_name
ORDER BY total_rides DESC

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_stations FROM citibike.gold_station_activity_v1

In [ ]:
%%sql
SELECT * FROM citibike.gold_station_activity_v1
ORDER BY total_rides DESC
LIMIT 10